In [1]:
##### Aligns sub-national labor data with national total
# defines functions for aligning data across each country
# runs functions and combines into final dataset

# uses shares from sub-regions from whatever year they are available (closest to 2020)
# but applies to national total in 2020 from ILO (assumes sub-regional distribution is the same)

import os
import pandas as pd

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
ILO_labor = pd.read_csv(f"{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Labor/subnational_labor_final.csv"

In [3]:
### Define function for aligning with national total 

def align_to_national (country_code):
    sub_labor = pd.read_csv(f"{cd}/Data/Clean/Labor/{country_code}_labor.csv")

    # pull national total based on ISO3
    country_labor = ILO_labor[ILO_labor['ISO3'] == country_code]

    # merge to sub_national labor
    sub_labor = sub_labor.merge(country_labor, on='ISO3', how='left')
    sub_labor['national_total_jobs'] = sub_labor['ag_labor_thousands_2020'] * 1000

    # calculate regional shares
    region_sum = sub_labor['ag_labor_jobs'].sum()
    sub_labor['region_sum'] = region_sum

    sub_labor['regional_share'] = sub_labor['ag_labor_jobs'] / sub_labor['region_sum']

    # apportion jobs 
    sub_labor['ag_jobs'] = sub_labor['national_total_jobs'] * sub_labor['regional_share']

    # add project ID
    sub_labor['PROJ_ID'] = sub_labor['ISO3'].astype(str) + '_' + sub_labor['GEO_ID'].astype(str)

    # select final column order 
    col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_jobs']
    sub_labor = sub_labor[col_to_keep]

    return sub_labor


In [4]:
#### Run US seperate because of country code issue

US_labor = pd.read_csv(f"{cd}/Data/Clean/Labor/USA_labor.csv")

# pull national total based on ISO3
country_labor = ILO_labor[ILO_labor['ISO3'] == 'USA']

# merge to sub_national labor
US_labor = US_labor.merge(country_labor, on='ISO3', how='left')
US_labor['national_total_jobs'] = US_labor['ag_labor_thousands_2020'] * 1000

# calculate regional shares
USA_sum = US_labor['ag_labor_jobs'].sum()
US_labor['region_sum'] = USA_sum

US_labor['regional_share'] = US_labor['ag_labor_jobs'] / US_labor['region_sum']

# apportion jobs 
US_labor['ag_jobs'] = US_labor['national_total_jobs'] * US_labor['regional_share']

# add project ID
US_labor["GEO_ID"] = US_labor["GEO_ID"].astype(str).str.zfill(5)
US_labor['PROJ_ID'] = US_labor['ISO3'].astype(str) + '_' + US_labor['GEO_ID'].astype(str)

# select final column order 
col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_jobs']
US_labor = US_labor[col_to_keep]

In [5]:
### Run alignment for all available countries

country_codes = ['BEL', 'BGR', 'BOL', 'BRA', 'CAN', 'CHN', 'DEU', 'ESP', 'FIN', 'FRA', 'GBR', 'GHA', 'GRC', 'HRV', 'HUN',
                 'ITA', 'JPN', 'MEX', 'NGA', 'NZL', 'PHL', 'POL', 'PRT', 'ROU',
                 'RUS', 'SWE', 'TUR', 'UGA', 'VNM']

aligned = {}
for code in country_codes:
    aligned[code] = align_to_national(code)

# concatinate all countries 
subnational_labor = pd.concat(aligned, ignore_index=True)

### Add in ZAF (because only subset of regions)
ZAF = pd.read_csv(f"{cd}/Data/Clean/Labor/ZAF_labor.csv")
ZAF.rename(columns={'ag_labor_jobs': 'ag_jobs'}, inplace=True)

ZAF['PROJ_ID'] = ZAF['ISO3'].astype(str) + '_' + ZAF['GEO_ID'].astype(str)
col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_jobs']
ZAF = ZAF[col_to_keep]

### Add in IND (because only subset of regions)
IND = pd.read_csv(f"{cd}/Data/Clean/Labor/IND_labor.csv")
IND.rename(columns={'ag_labor_jobs': 'ag_jobs'}, inplace=True)

IND['PROJ_ID'] = IND['ISO3'].astype(str) + '_' + IND['GEO_ID'].astype(str)
col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_jobs']
IND = IND[col_to_keep]

subnational_labor = pd.concat([subnational_labor, ZAF, IND, US_labor], ignore_index=True)
subnational_labor = subnational_labor.dropna()

In [6]:
# Save
subnational_labor.to_csv(save_path, index=False)